# Cellule 1 — Setup, config, modèle

In [ ]:
# ── CELLULE 1 : Setup, config, modèle ──────────────────────────────────────
!pip install -q huggingface_hub h5py torch tqdm scipy

import os, torch, h5py, json, shutil
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm
from scipy.ndimage import distance_transform_edt
from huggingface_hub import login, HfApi, snapshot_download, hf_hub_download, list_repo_files
import torch.nn as nn

# ── Auth ────────────────────────────────────────────────────────────────────
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')
login(HF_TOKEN)

# ── Repos ───────────────────────────────────────────────────────────────────
REPO_DATA    = "Yousra0x0/validation_20_preprocessed"   # .h5 déjà preprocessés (lecture)
REPO_MODEL   = "hourouu/model4"                          # poids UNet (public, lecture)
REPO_RESULTS = "Yousra0x0/model4_validation_20CTCOVIDE"           # résultats (écriture)

# ── Params ──────────────────────────────────────────────────────────────────
THRESHOLD = 0.5
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
DIR_H5      = "/content/val_h5"
DIR_RESULTS = "/content/val_results"
os.makedirs(DIR_H5,      exist_ok=True)
os.makedirs(DIR_RESULTS, exist_ok=True)

api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=REPO_RESULTS, repo_type="dataset", exist_ok=True)
print(f"Repo résultats prêt : https://huggingface.co/datasets/{REPO_RESULTS}")
print(f"Device : {DEVICE}")

# ── Architecture UNetV2 ─────────────────────────────────────────────────────
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm2d(out_ch, affine=True), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm2d(out_ch, affine=True), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)

class UNetV2(nn.Module):
    def __init__(self, in_ch=1, base_ch=32):
        super().__init__()
        b = base_ch
        self.enc1 = ConvBlock(in_ch, b);   self.enc2 = ConvBlock(b,   b*2)
        self.enc3 = ConvBlock(b*2,  b*4);  self.enc4 = ConvBlock(b*4, b*8)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = ConvBlock(b*8, b*16)
        self.up4  = nn.ConvTranspose2d(b*16, b*8, 2, stride=2)
        self.dec4 = ConvBlock(b*16, b*8)
        self.up3  = nn.ConvTranspose2d(b*8,  b*4, 2, stride=2)
        self.dec3 = ConvBlock(b*8,  b*4)
        self.up2  = nn.ConvTranspose2d(b*4,  b*2, 2, stride=2)
        self.dec2 = ConvBlock(b*4,  b*2)
        self.up1  = nn.ConvTranspose2d(b*2,  b,   2, stride=2)
        self.dec1 = ConvBlock(b*2,  b)
        self.out  = nn.Conv2d(b, 1, 1)

    def forward(self, x):
        s1 = self.enc1(x);               s2 = self.enc2(self.pool(s1))
        s3 = self.enc3(self.pool(s2));   s4 = self.enc4(self.pool(s3))
        b  = self.bottleneck(self.pool(s4))
        d4 = self.dec4(torch.cat([self.up4(b),  s4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), s3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), s2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), s1], dim=1))
        return self.out(d1)
# ── Chargement des poids ────────────────────────────────────────────────────
model = UNetV2(in_ch=1, base_ch=32).to(DEVICE)
ckpt_path = None
for fname in ["best.pth", "latest.pth"]:
    try:
        ckpt_path = hf_hub_download(repo_id=REPO_MODEL, filename=fname, repo_type="dataset")
        print(f"Poids trouvés : {fname}")
        break
    except Exception as e:
        print(f"  {fname} : {e}")

if ckpt_path is None:
    raise FileNotFoundError(f"Aucun poids trouvé dans {REPO_MODEL}")

ckpt = torch.load(ckpt_path, map_location=DEVICE)
if isinstance(ckpt, dict):
    state = (ckpt.get("model_state") or ckpt.get("model_state_dict")
             or ckpt.get("state_dict") or ckpt)
    if "model_state" in ckpt:
        print(f"  epoch={ckpt.get('epoch','?')}  best_dice={ckpt.get('best_dice',0):.4f}")
else:
    state = ckpt

model.load_state_dict(state, strict=True)
model.eval()
print(f"\nUNetV2 chargé sur {DEVICE}  —  {sum(p.numel() for p in model.parameters()):,} params")

Repo résultats prêt : https://huggingface.co/datasets/Yousra0x0/model4_validation_20CTCOVIDE
Device : cuda


best.pth:   0%|          | 0.00/93.2M [00:00<?, ?B/s]

Poids trouvés : best.pth
  epoch=7  best_dice=0.9044

UNetV2 chargé sur cuda  —  7,762,465 params


# Cellule 2 — Inférence + métriques

In [ ]:
# ── CELLULE 2 : Inférence + métriques ──────────────────────────────────────

# ── Fonctions métriques ─────────────────────────────────────────────────────
def dice_score(pred, target, eps=1e-6):
    inter = (pred * target).sum()
    return float((2*inter + eps) / (pred.sum() + target.sum() + eps))

def iou_score(pred, target, eps=1e-6):
    inter = (pred * target).sum()
    return float((inter + eps) / (pred.sum() + target.sum() - inter + eps))

def precision_recall_f1(pred, target, eps=1e-6):
    tp = (pred * target).sum()
    fp = (pred * (1 - target)).sum()
    fn = ((1 - pred) * target).sum()
    prec = float((tp + eps) / (tp + fp + eps))
    rec  = float((tp + eps) / (tp + fn + eps))
    f1   = float(2 * prec * rec / (prec + rec + eps))
    return prec, rec, f1

def sensitivity_specificity(pred, target, eps=1e-6):
    tp = (pred * target).sum()
    fp = (pred * (1 - target)).sum()
    fn = ((1 - pred) * target).sum()
    tn = ((1 - pred) * (1 - target)).sum()
    return float((tp+eps)/(tp+fn+eps)), float((tn+eps)/(tn+fp+eps))

def hausdorff_95(pred, target):
    pb, tb = pred.astype(bool), target.astype(bool)
    if not pb.any() or not tb.any():
        return float("nan")
    try:
        d_p2t = distance_transform_edt(~tb)[pb]
        d_t2p = distance_transform_edt(~pb)[tb]
        return float(np.percentile(np.concatenate([d_p2t, d_t2p]), 95))
    except Exception:
        return float("nan")

def volume_error(pred, target):
    vt = target.sum()
    return float("nan") if vt == 0 else float(abs(pred.sum() - vt) / vt * 100)

def all_metrics(pred, target):
    pred, target = pred.astype(np.float32), target.astype(np.float32)
    prec, rec, f1 = precision_recall_f1(pred, target)
    sens, spec    = sensitivity_specificity(pred, target)
    return {
        "dice":        dice_score(pred, target),
        "iou":         iou_score(pred, target),
        "precision":   prec, "recall": rec, "f1": f1,
        "sensitivity": sens, "specificity": spec,
        "hd95":        hausdorff_95(pred, target),
        "vol_error_%": volume_error(pred, target),
    }

# ── Download des .h5 depuis Yousra0x0/validation_20_preprocessed ────────────
print(f"Téléchargement de {REPO_DATA} ...")
snapshot_download(
    repo_id=REPO_DATA,
    repo_type="dataset",
    local_dir=DIR_H5,
    token=HF_TOKEN,
    ignore_patterns=["*.md", "*.parquet", "*.arrow"]
)

h5_files = sorted(Path(DIR_H5).rglob("*.h5"))
print(f"  {len(h5_files)} fichiers .h5 trouvés\n")

# ── Inférence slice-by-slice ────────────────────────────────────────────────
all_results  = {}   # uid -> dict métriques
all_metrics_list = []

model.eval()
for h5_path in tqdm(h5_files, desc="Validation"):
    uid = h5_path.stem
    try:
        with h5py.File(h5_path, "r") as fh:
            # Supporte les deux conventions de nommage
            img_key  = "image" if "image" in fh else "img"
            msk_key  = "mask"  if "mask"  in fh else "msk"
            image = fh[img_key][:]   # (N, H, W) float32, déjà [0,1]
            mask  = fh[msk_key][:]   # (N, H, W) binaire

        pred_vol = np.zeros_like(mask, dtype=np.float32)
        with torch.no_grad():
            for i in range(image.shape[0]):
                slc  = torch.from_numpy(image[i]).unsqueeze(0).unsqueeze(0).float().to(DEVICE)
                prob = torch.sigmoid(model(slc)).cpu().numpy()[0, 0]
                pred_vol[i] = (prob > THRESHOLD).astype(np.float32)

        m = all_metrics(pred_vol, mask)
        all_results[uid]      = m
        all_metrics_list.append(m)
        print(f"  [{uid[:40]:40s}]  Dice={m['dice']:.4f}  IoU={m['iou']:.4f}  HD95={m['hd95']:.1f}")

    except Exception as e:
        print(f"  ERREUR [{uid}] : {e}")

print(f"\n{len(all_results)}/{len(h5_files)} volumes validés")

Téléchargement de Yousra0x0/validation_20_preprocessed ...


Fetching 22 files:   0%|          | 0/22 [00:00<?, ?it/s]

  20 fichiers .h5 trouvés



Validation:   0%|          | 0/20 [00:00<?, ?it/s]

  [coronacases_001                         ]  Dice=0.9574  IoU=0.9183  HD95=0.0
  [coronacases_002                         ]  Dice=0.9626  IoU=0.9279  HD95=0.0
  [coronacases_003                         ]  Dice=0.8984  IoU=0.8155  HD95=1.4
  [coronacases_004                         ]  Dice=0.9672  IoU=0.9364  HD95=0.0
  [coronacases_005                         ]  Dice=0.9643  IoU=0.9310  HD95=0.0
  [coronacases_006                         ]  Dice=0.9626  IoU=0.9279  HD95=0.0
  [coronacases_007                         ]  Dice=0.9578  IoU=0.9190  HD95=0.0
  [coronacases_008                         ]  Dice=0.9637  IoU=0.9299  HD95=0.0
  [coronacases_009                         ]  Dice=0.9598  IoU=0.9227  HD95=0.0
  [coronacases_010                         ]  Dice=0.9423  IoU=0.8909  HD95=1.0
  [radiopaedia_10_85902_1                  ]  Dice=0.9381  IoU=0.8835  HD95=1.0
  [radiopaedia_10_85902_3                  ]  Dice=0.9522  IoU=0.9087  HD95=0.0
  [radiopaedia_14_85914_0               

# Cellule 3 — Résumé + upload vers Yousra0x0/unet_validation_20

In [ ]:
# ── CELLULE 3 : Résumé + upload ─────────────────────────────────────────────
from datetime import datetime

def agg(key):
    vals = [m[key] for m in all_metrics_list
            if not (isinstance(m[key], float) and np.isnan(m[key]))]
    if not vals:
        return {"mean": 0, "std": 0, "min": 0, "max": 0, "median": 0}
    return {
        "mean":   float(np.mean(vals)),   "std":    float(np.std(vals)),
        "min":    float(np.min(vals)),    "max":    float(np.max(vals)),
        "median": float(np.median(vals)),
    }

METRIC_KEYS = ["dice", "iou", "f1", "precision", "recall",
               "sensitivity", "specificity", "hd95", "vol_error_%"]

summary = {
    "model":      REPO_MODEL,
    "data":       REPO_DATA,
    "n_volumes":  len(all_metrics_list),
    "threshold":  THRESHOLD,
    "timestamp":  datetime.now().isoformat(),
    **{k: agg(k) for k in METRIC_KEYS},
}

# ── Affichage console ────────────────────────────────────────────────────────
print(f"\n{'='*62}")
print(f"  RÉSULTATS FINAUX  ({summary['n_volumes']} volumes)")
print(f"{'='*62}")
print(f"  {'Métrique':<18} {'Mean':>8} {'±Std':>8}  {'Min':>8} – {'Max':>8}")
print(f"  {'-'*56}")
for k in METRIC_KEYS:
    s = summary[k]
    print(f"  {k:<18} {s['mean']:>8.4f} {s['std']:>8.4f}  {s['min']:>8.4f} – {s['max']:>8.4f}")
print(f"{'='*62}")

# ── Sauvegarde JSON ──────────────────────────────────────────────────────────
full_results  = {"summary": summary, "per_patient": all_results}
json_path     = os.path.join(DIR_RESULTS, "validation_results.json")
with open(json_path, "w") as f:
    json.dump(full_results, f, indent=2)
print(f"\nJSON : {json_path}")

# ── Sauvegarde rapport TXT ───────────────────────────────────────────────────
report_path = os.path.join(DIR_RESULTS, "validation_report.txt")
with open(report_path, "w") as f:
    f.write("=" * 62 + "\n")
    f.write("  VALIDATION REPORT — UNet2D (hourouu/model4)\n")
    f.write(f"  Date      : {summary['timestamp']}\n")
    f.write(f"  Volumes   : {summary['n_volumes']}\n")
    f.write(f"  Threshold : {summary['threshold']}\n")
    f.write(f"  Model     : {summary['model']}\n")
    f.write(f"  Data      : {summary['data']}\n")
    f.write("=" * 62 + "\n\n")
    f.write(f"{'Metric':<20} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10} {'Median':>10}\n")
    f.write("-" * 68 + "\n")
    for k in METRIC_KEYS:
        s = summary[k]
        f.write(f"{k:<20} {s['mean']:>10.4f} {s['std']:>10.4f} "
                f"{s['min']:>10.4f} {s['max']:>10.4f} {s['median']:>10.4f}\n")
    f.write("\n\n--- Résultats par patient ---\n\n")
    for pid, m in all_results.items():
        f.write(f"[{pid}]\n")
        for mk, mv in m.items():
            f.write(f"  {mk:<15} = {mv:.6f}\n")
        f.write("\n")
print(f"TXT  : {report_path}")

# ── Upload vers Yousra0x0/unet_validation_20 ─────────────────────────────────
print(f"\nUpload vers {REPO_RESULTS} ...")
api.create_repo(repo_id=REPO_RESULTS, repo_type="dataset", exist_ok=True)

for fpath in [json_path, report_path]:
    if os.path.exists(fpath):
        fname = os.path.basename(fpath)
        try:
            api.upload_file(
                path_or_fileobj=fpath,
                path_in_repo=f"validation/{fname}",
                repo_id=REPO_RESULTS,
                repo_type="dataset",
                commit_message=f"Validation UNet — {summary['n_volumes']} volumes"
            )
            print(f"  ✓ {fname}")
        except Exception as e:
            print(f"  ✗ {fname} : {e}")
    else:
        print(f"  ⚠ Absent : {fpath}")

print(f"\nPIPELINE TERMINÉ")
print(f"  Résultats → https://huggingface.co/datasets/{REPO_RESULTS}")


  RÉSULTATS FINAUX  (20 volumes)
  Métrique               Mean     ±Std       Min –      Max
  --------------------------------------------------------
  dice                 0.9499   0.0169    0.8984 –   0.9672
  iou                  0.9051   0.0299    0.8155 –   0.9364
  f1                   0.9499   0.0169    0.8984 –   0.9672
  precision            0.9539   0.0075    0.9424 –   0.9664
  recall               0.9466   0.0328    0.8537 –   0.9832
  sensitivity          0.9466   0.0328    0.8537 –   0.9832
  specificity          0.9926   0.0030    0.9861 –   0.9976
  hd95                 0.3707   0.5124    0.0000 –   1.4142
  vol_error_%          2.6402   2.5781    0.1341 –   9.9448

JSON : /content/val_results/validation_results.json
TXT  : /content/val_results/validation_report.txt

Upload vers Yousra0x0/model4_validation_20CTCOVIDE ...
  ✓ validation_results.json
  ✓ validation_report.txt

PIPELINE TERMINÉ
  Résultats → https://huggingface.co/datasets/Yousra0x0/model4_validation_20